# ChefLM — Chat with Chef

Download a pre-trained ~7M parameter milkshake-chef LLM and chat with it. Just run all cells.

**Model:** `BT-Rajan/chef-9m` (placeholder — create and upload this repo first, or set `HF_REPO` to wherever you've published your trained checkpoint)

In [ ]:
# Setup + Download
!pip install -q torch tokenizers huggingface_hub
import os, shutil
if os.path.exists('/content/chef'): shutil.rmtree('/content/chef')
os.makedirs('/content/chef'); os.chdir('/content/chef')

HF_REPO = os.environ.get('HF_REPO', 'BT-Rajan/chef-9m')  # placeholder — create this repo first

from huggingface_hub import snapshot_download
snapshot_download(repo_id=HF_REPO, local_dir='.')
print('Model downloaded.')

In [ ]:
# Load model
from inference import ChefInference
import torch

engine = ChefInference('pytorch_model.bin', 'tokenizer.json',
                        device='cuda' if torch.cuda.is_available() else 'cpu')

def chat(prompt, lang='en'):
    # lang: 'en' or 'ar' — forces the reply language regardless of
    # what script `prompt` itself is written in.
    return engine.chat_completion(
        [{'role': 'user', 'content': prompt}], max_tokens=64, lang=lang
    )['choices'][0]['message'].get('content', '').strip()

# Quick test — English, then the same idea in Arabic
for p in ['what is your favorite milkshake flavor', 'how do you make a milkshake',
          'is a milkshake healthy', 'what toppings go well on a milkshake']:
    print(f'You> {p}\nChef> {chat(p)}\n')

for p in ['ما هي مكونات الميلك شيك', 'كيف تصنع ميلك شيك']:
    print(f'You> {p}\nChef> {chat(p, lang="ar")}\n')

In [ ]:
# Interactive chat — type your messages
# Set LANG to 'ar' first if you'd rather chat in Arabic.
LANG = 'en'
while True:
    try:
        p = input('You> ').strip()
    except (KeyboardInterrupt, EOFError):
        break
    if not p or p.lower() in ('quit', 'exit', 'q'):
        print("Chef> bye. i'll be here whenever you want to talk milkshakes."); break
    print(f'Chef> {chat(p, lang=LANG)}\n')